# 1.Introdução

## Engenharia de Variaveis Pipeline

Neste notebook, será estudado aplicado técnicas de engenharia de featura que foram selecionadas após a etapa de EDA, aqui será criado todo o pre-processamento e aplicação dentro de um pipeline, buscando otimizar a execução e a legibilidade do modelo, também será muito util para o deploy, posteriormente,

# 2. Escopo e Estratégia de Engenharia de Variáveis

Nesta etapa inicial da engenharia de variáveis, o foca será a base 'application_train.csv', considerando um escopo incremental para o projeto.

As principais diretrizes adotadas incluem:
- Manutenção de variáveis com potencial informativo
- Tratamento consicente de valores ausentes
- Preparação das variáveis para modelos lineares
- Priorização de interpretabilidade

# 3. Importação das bibliotecas e configurações iniciais

In [63]:
import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# 4. Carregamento dos dados 
Os dados são carregados a partir  da pasta 'data/' que não é versionada no repositorio seguindo boas práticas da engenharia de dados.

In [78]:
application_train = pd.read_csv('../data/application_train.csv', encoding = 'iso-8859-1')
application_train.shape

(307511, 122)

# 5. Separação as Features e o Target
Nesta seção, a variável target ('TARGET') é separada das variáveis explicativas, garantido clareza no pipeline de preparação dos dados e evitando vazamento de informação.

In [114]:
# SEPARANDO AS FEATURES E O TARGET
X = application_train.drop(columns = ['TARGET'])
Y = application_train['TARGET']

# EXIBINDO O FORMATO DAS FEATURE E DO TARGET
print('features: {},\ntarget: {}'.format(X.shape, Y.shape))

# COPIANDO O RESULTADO PARA MODIFICAÇÕES POSTERIORES
features = X.copy()
target = Y.copy()

features: (307511, 121),
target: (307511,)


# 6. Separação das Variaveis

Nesta etapa, pós fazer a separação das features e do target. Estarei separando as varaiveis por tipos (float, int, categoricas)

In [115]:
# EXIBINDO QUAIS OS TIPOS DAS FEATURES
features.dtypes.unique()

# SEPARANDO AS FEATURES ENTRE NUMERICAS E CATEGORICAS
numericas = features.select_dtypes(['int64','float64']).columns
categoricas = features.select_dtypes(['O']).columns

# EXIBINDO O FORMATO DE CADA TIPO CRIADO
print('numericas: {}\ncategoricas: {}'.format(numericas.shape, categoricas.shape))

numericas: (105,)
categoricas: (16,)


# 7. Criando o pipeline

# MISSING FLAG TRANSFORMER

In [120]:
class MissingFlagTransformer(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):

        self.colunas_com_missing = (
            X.columns[
                X.isnull().mean() > 0
            ]
        )

        return self

    def transform(self, X):

        X = X.copy()

        for coluna in self.colunas_com_missing:

            X[f'{coluna}_missing_flag'] = (
                X[coluna]
                .isnull()
                .astype(int)
            )

        return X

# Feature EngineeringTransformer

In [121]:
class FeatureEngineeringTransformer(
    BaseEstimator,
    TransformerMixin
):

    def fit(self, X, y=None):

        return self

    def transform(self, X):

        X = X.copy()

        # =====================================================
        # LOG TRANSFORM
        # =====================================================

        X['AMT_INCOME_TOTAL_log'] = np.log1p(
            X['AMT_INCOME_TOTAL']
        )

        X['AMT_CREDIT_log'] = np.log1p(
            X['AMT_CREDIT']
        )

        X['AMT_GOODS_PRICE_log'] = np.log1p(
            X['AMT_GOODS_PRICE']
        )

        X['AMT_ANNUITY_log'] = np.log1p(
            X['AMT_ANNUITY']
        )

        # =====================================================
        # EVITANDO DIVISÃO POR ZERO
        # =====================================================

        X['CNT_FAM_MEMBERS'] = (
            X['CNT_FAM_MEMBERS']
            .replace(0, 1)
        )

        # =====================================================
        # FEATURE CREATION
        # =====================================================

        # Quanto da renda está comprometida com crédito

        X['RATIO_CREDIT_INCOME'] = (
            X['AMT_CREDIT']
            /
            (X['AMT_INCOME_TOTAL'] + 1)
        )

        # Esforço mensal de pagamento

        X['RATIO_ANNUITY_INCOME'] = (
            X['AMT_ANNUITY']
            /
            (X['AMT_INCOME_TOTAL'] + 1)
        )

        # Crédito por membro familiar

        X['CREDIT_PER_PERSON'] = (
            X['AMT_CREDIT']
            /
            (X['CNT_FAM_MEMBERS'] + 1)
        )

        # Renda por membro familiar

        X['INCOME_PER_PERSON'] = (
            X['AMT_INCOME_TOTAL']
            /
            (X['CNT_FAM_MEMBERS'] + 1)
        )

        # Relação crédito x bens

        X['RATIO_CREDIT_GOODS'] = (
            X['AMT_CREDIT']
            /
            (X['AMT_GOODS_PRICE'] + 1)
        )

        return X

In [122]:
# pipeline numerico
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'median'))
])

#pipeline categorico
categoric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy = 'constant', fill_value = "Unknown")),
    ('encoder', OneHotEncoder(drop = 'first',handle_unknown = 'ignore'))
])

# aplicas as transformações aplicadas pelo pipeline dentro das colunas númericas e cateoricas
preprocessor = ColumnTransformer(
    transformers = [
        ('numericas', numeric_pipeline, numericas)
    ,
    ('categoricas', categoric_pipeline, categoricas)
])


pipeline_preprocessamento = Pipeline(

    steps=[

        (
            'missing_flags',
            MissingFlagTransformer()
        ),

        (
            'feature_engineering',
            FeatureEngineeringTransformer()
        ),

        (
            'preprocessor',
            preprocessor
        )

    ]

)

In [123]:
features_transformadas = (
    pipeline_preprocessamento
    .fit_transform(features)
)
print(features_transformadas.shape)

(307511, 235)


In [124]:
features_fe = (
    FeatureEngineeringTransformer()
    .fit_transform(
        MissingFlagTransformer()
        .fit_transform(
            features.copy()
        )
    )
)

print(features_fe.shape)

(307511, 197)


In [125]:
X_final = preprocessor.fit_transform(
    features_fe
)

print(X_final.shape)

(307511, 235)
